In [0]:
dbutils.library.restartPython()

In [ ]:
import json
import ast

In [ ]:
dbutils.widgets.text("table_config", "{}")
table_config_json = dbutils.widgets.get("table_config")

print(f"Raw parameter received (first 200 chars): {table_config_json[:200]}...")

table_configs = json.loads(table_config_json)
print(f"\n✓ Processing table: {table_configs.get('source_table', 'UNKNOWN')}")
print(f"✓ Report type: {table_configs.get('report_type', 'UNKNOWN')}")
print(f"✓ Filters column: {table_configs.get('filters_column', 'None')}")
print(f"✓ Filters column type: {table_configs.get('filters_column_type', 'None')}")

In [ ]:
from databricks.labs.lakebridge.config import (
    DatabaseConfig,
    ReconcileConfig,
    ReconcileMetadataConfig
)
from databricks.labs.lakebridge.reconcile.recon_config import Filters, Table, Transformation
from databricks.labs.lakebridge.reconcile.exception import ReconciliationException
from databricks.labs.lakebridge.reconcile.trigger_recon_service import TriggerReconService
from databricks.sdk import WorkspaceClient
from databricks.labs.lakebridge import __version__
from dataclasses import dataclass

@dataclass
class TableRecon:
    source_schema: str
    target_catalog: str
    target_schema: str
    tables: list[Table]
    source_catalog: str | None = None

ws = WorkspaceClient(product="lakebridge", product_version=__version__)
print(" Libraries imported and workspace client initialized")

In [ ]:
try:
    # Build reconcile config
    reconcile_config = ReconcileConfig(
        data_source=table_configs["data_source"],
        report_type=table_configs["report_type"].lower(),
        secret_scope=table_configs["secret_scope"],

        database_config=DatabaseConfig(
            source_catalog=table_configs["source_catalog"],
            source_schema=table_configs["source_schema"],
            target_catalog=table_configs["target_catalog"],
            target_schema=table_configs["target_schema"]
        ),

        metadata_config=ReconcileMetadataConfig(
            catalog=table_configs["target_catalog"],
            schema="lakebridge_recon"
        ))

    print(" Reconcile config created")

    # Handle filters dynamically based on filters_column_type from CSV
    filters_col = table_configs.get('filters_column')
    filters_col_type = str(table_configs.get('filters_column_type', '')).lower().strip()
    table_filters = None
    
    if filters_col and str(filters_col).strip().lower() not in ['nan', 'none', '', 'null']:
        source_condition = str(table_configs.get('source_filters_condition', '')).strip()
        target_condition = str(table_configs.get('target_filters_condition', '')).strip()
        
        if source_condition and target_condition:
            if filters_col_type == 'timestamp':
                # Timestamp columns - use special timestamp handling
                target_ts_expr = f"coalesce(try_to_timestamp({filters_col}), to_timestamp(regexp_replace(trim({filters_col}), ' +', ' '), 'MMM d yyyy h:mma'))"
                table_filters = Filters(
                    source=f"{filters_col} {source_condition}",
                    target=f"{target_ts_expr} {target_condition}"
                )
                print(f" Applied timestamp filter on column: {filters_col}")
            elif filters_col_type == 'string':
                # String columns - use LOWER() for case-insensitive comparison
                table_filters = Filters(
                    source=f"lower({filters_col}) {source_condition}",
                    target=f"lower({filters_col}) {target_condition}"
                )
                print(f" Applied string filter on column: {filters_col}")
            elif filters_col_type == 'numeric':
                # Numeric columns - direct comparison
                table_filters = Filters(
                    source=f"{filters_col} {source_condition}",
                    target=f"{filters_col} {target_condition}"
                )
                print(f" Applied numeric filter on column: {filters_col}")
            else:
                # Default: direct comparison without transformation
                table_filters = Filters(
                    source=f"{filters_col} {source_condition}",
                    target=f"{filters_col} {target_condition}"
                )
                print(f" Applied default filter on column: {filters_col}")
        else:
            print(f" Skipping filter - missing conditions for column: {filters_col}")
    else:
        print(" No filters configured for this table")


    recon_transformations = []
    transformation_str = table_configs.get('transformation', '')
    
    print("\n" + "="*80)
    print("TRANSFORMATION LOADING:")
    print("="*80)
    
    if transformation_str and str(transformation_str).strip().lower() not in ['', 'nan', 'none', 'null']:
        try:
            # Parse as JSON from CSV
            transformation_dict = json.loads(transformation_str)
            
            # Convert to Transformation objects
            for col_name, transforms in transformation_dict.items():
                recon_transformations.append(
                    Transformation(
                        column_name=col_name,
                        source=transforms['source'],
                        target=transforms['target']
                    )
                )
            
            print(f" Loaded {len(recon_transformations)} transformations from CSV for {table_configs['source_table']}")
            for trans in recon_transformations:
                print(f"\n  Column: {trans.column_name}")
                print(f"    Source: {trans.source[:100]}...")
                print(f"    Target: {trans.target[:100]}...")
                
        except Exception as e:
            print(f" Warning: Could not parse transformations from CSV: {e}")
            print(f"   Transformation string (first 200 chars): {transformation_str[:200]}...")
    else:
        print(f" No transformations configured for {table_configs['source_table']}")
    
    print("="*80 + "\n")

    # Build table recon config
    table_recon = TableRecon(
        source_schema=table_configs["source_schema"],
        target_catalog=table_configs["target_catalog"],
        target_schema=table_configs["target_schema"],
        tables=[
            Table(
                source_name=table_configs["source_table"],
                target_name=table_configs["target_table"],
                join_columns=table_configs["join_columns"],  # Already a list from config
                filters=table_filters,
                transformations=recon_transformations if recon_transformations else None
            )
        ]
    )

    print(" Table recon config created")
    print(f" Starting reconciliation for {table_configs['source_table']}...")

    # Trigger reconciliation
    result = TriggerReconService.trigger_recon(
        ws=ws,
        spark=spark,
        table_recon=table_recon,
        reconcile_config=reconcile_config
    )
    
    print(f"\n{'='*60}")
    print(f" RECON SUCCESS for {table_configs['source_table']}")
    print(f" Recon ID: {result.recon_id}")
    print(f"{'='*60}\n")
    
except ReconciliationException as re:
    
    recon_output = re.args[1] if len(re.args) > 1 else None
    if recon_output:
        print(f"\n{'='*60}")
        print(f" RECON COMPLETED WITH MISMATCHES for {table_configs['source_table']}")
        print(f" Recon ID: {recon_output.recon_id}")
        
        for result in recon_output.results:
            print(f"\n  Source: {result.source_table_name}")
            print(f"  Target: {result.target_table_name}")
            print(f"  Row Count Match: {' PASS' if result.status.row else ' FAIL'}")
            print(f"  Column Values Match: {' PASS' if result.status.column else ' FAIL'}")
            print(f"  Schema Match: {' PASS' if result.status.schema else ' FAIL'}")
            if result.status.aggregate is not None:
                print(f"  Aggregate Match: {' PASS' if result.status.aggregate else ' FAIL'}")
    
except Exception as e:
    print(f"\n{'='*60}")
    print(f"✗ RECON FAILED for {table_configs.get('source_table', 'UNKNOWN')}")
    print(f"✗ Error: {e}")
    print(f"{'='*60}\n")
    raise